# Práctica 1: Transformación de formatos a coordenadas tridimensionales
## Grafos moleculares y representaciones químicas

**Bloque temático**: 1 – Generación de estructuras y espacio conformacional  
**Número de práctica**: 1  
**Nivel de dificultad**: Básico  
**Tiempo estimado**: 1 h (semilla) + 2 h (bosque y análisis)  
**Pipeline central**: SMILES → grafo molecular → geometría 3D (ETKDG) → pre-opt FF (MMFF94) → opt semiempírica (GFN2-xTB) → dataset estructural

---

## Introducción

Antes de realizar cualquier cálculo cuántico, una molécula debe existir como un objeto computacional con coordenadas tridimensionales explícitas. Este paso, aparentemente trivial, encierra decisiones metodológicas con consecuencias directas sobre la calidad de todos los cálculos posteriores: una geometría inicial mal construida puede converger a un mínimo incorrecto, producir frecuencias imaginarias espurias o generar resultados numéricos que no corresponden a ninguna especie química real.

La representación más común en quimioinformática es el **SMILES** (*Simplified Molecular Input Line Entry System*), una cadena de texto que codifica la conectividad molecular mediante reglas gramaticales precisas. Un SMILES como `CC(=O)Nc1ccccc1` describe sin ambigüedad la acetanilida; sin embargo, no contiene información sobre los ángulos de enlace, las longitudes de enlace ni la disposición espacial de los átomos. Pasar de esa cadena lineal a un conjunto de coordenadas cartesianas $(x_i, y_i, z_i)$ es el problema que resuelve esta práctica.

Las herramientas modernas —principalmente RDKit y OpenBabel— implementan algoritmos de incrustación tridimensional que combinan reglas geométricas empíricas, campos de fuerza y, en el caso del algoritmo ETKDG (*Experimental-Torsion distance geometry with basic Knowledge*), distribuciones de ángulos diedros extraídas de la Cambridge Structural Database. El resultado es una geometría de partida razonable en pocos milisegundos, sin importar el tamaño de la molécula.

Una vez disponible la geometría inicial, la práctica introduce el primer pipeline del manual: pre-optimización con un campo de fuerza (MMFF94) seguida de optimización semiempírica con GFN2-xTB. Este protocolo de dos pasos es suficiente para la mayoría de las moléculas orgánicas pequeñas y medianas, y constituye el punto de partida estándar para los cálculos de estructura electrónica del Bloque 2.

En términos del modelo Semilla–Bosque: la **semilla** es la construcción y optimización de una molécula sencilla que el estudiante ejecuta por sí mismo. El **bosque** es un dataset de 50 moléculas orgánicas con diversidad estructural —aromáticos, heteroaromáticos, sistemas flexibles y casos difíciles— para los cuales el mismo pipeline ya fue ejecutado. El análisis del bosque permite identificar qué características estructurales hacen que la incrustación 3D sea más o menos confiable, y cómo la energía de pre-optimización se correlaciona con descriptores topológicos básicos.

### Pipeline visual

```
SMILES (texto)
     ↓ RDKit
Grafo molecular (conectividad)
     ↓ ETKDG
Geometría 3D (incrustación)
     ↓ MMFF94
Pre-optimización (campo de fuerza)
     ↓ GFN2-xTB
Optimización semiempírica
     ↓ Análisis
Dataset estructural
```

## Marco teórico

### Conceptos clave

**1. Representaciones moleculares y su jerarquía informacional.**  
Una molécula puede codificarse en distintos niveles de detalle:
- **0D**: Fórmula molecular (solo composición)
- **1D**: SMILES o InChI (conectividad y orden de enlace)
- **2D**: Grafo 2D (estereoquímica plana)
- **3D**: Coordenadas cartesianas (geometría completa)

Las coordenadas cartesianas 3D son las únicas que permiten calcular propiedades que dependen de la geometría (energías, frecuencias, densidades electrónicas).

**2. Geometría de distancias (DG) e incrustación 3D.**  
El algoritmo ETKDG convierte un grafo molecular en coordenadas cartesianas resolviendo un problema de optimización sobre matrices de distancias interatómicas.

**3. Campos de fuerza (FF).**  
MMFF94 es una función empírica que describe la energía como suma de contribuciones de enlace, ángulo, torsión e interacciones no covalentes.

**4. GFN2-xTB.**  
Método semiempírico de enlace fuerte que reproduce geometrías con precisión comparable a DFT/def2-SVP pero 3–4 órdenes de magnitud más rápido.

**5. RMSD como métrica de similitud.**  
$$\mathrm{RMSD} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}\left|\mathbf{r}_i^{(A)} - \mathbf{r}_i^{(B)}\right|^2}$$

## Instalación de dependencias

Para ejecutar esta práctica necesitas instalar las siguientes librerías. Si estás en **Google Colab**, ejecuta la celda siguiente:

In [ ]:
# Solo para Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install rdkit numpy scipy pandas matplotlib seaborn py3Dmol -q
    print("✓ Dependencias instaladas en Colab")
else:
    print("✓ Ejecutando en entorno local o Jupyter")

## Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdMolDescriptors

# Configuración de visualización
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 11

print("✓ Todas las librerías importadas correctamente")

## Experimento semilla: Cafeína

La semilla de esta práctica es la **cafeína** (`Cn1cnc2c1c(=O)n(c(=O)n2C)C`, CAS 58-08-2), elegida porque:

1. Tiene 24 átomos pesados — pequeña pero no trivial
2. Posee un núcleo heterocíclico aromático y grupos carbonilo
3. Su estructura cristalina está bien determinada por difracción de rayos X
4. Es una molécula que todos los estudiantes conocen

### Paso 1: Construir el grafo molecular

In [ ]:
# SMILES para cafeína
smiles = 'Cn1cnc2c1c(=O)n(c(=O)n2C)C'
nombre = 'cafeina'

# Crear objeto molecular
mol = Chem.MolFromSmiles(smiles)
if mol is None:
    raise ValueError(f'SMILES inválido: {smiles}')

# Información del grafo
print(f'Átomos totales:    {mol.GetNumAtoms()}')
print(f'Átomos pesados:    {mol.GetNumHeavyAtoms()}')
print(f'Fórmula:           {rdMolDescriptors.CalcMolFormula(mol)}')
print(f'Peso molecular:    {rdMolDescriptors.CalcExactMolWt(mol):.4f} Da')
print(f'Enlaces rotativos: {rdMolDescriptors.CalcNumRotatableBonds(mol)}')

### Paso 2: Visualizar estructura 2D

In [ ]:
# Generar representación 2D
img = Draw.MolToImage(mol, size=(400, 300))
plt.figure(figsize=(6, 5))
plt.imshow(img)
plt.axis('off')
plt.title('Estructura 2D de cafeína', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Paso 3: Incrustación 3D con ETKDG

In [ ]:
# Añadir hidrógenos explícitos
mol_h = Chem.AddHs(mol)

# Parámetros ETKDG
params = AllChem.ETKDGv3()
params.randomSeed = 42  # Reproducibilidad

# Realizar incrustación
ok = AllChem.EmbedMolecule(mol_h, params)

if ok == -1:
    print('✗ Incrustación fallida')
else:
    print('✓ Incrustación exitosa')
    
    # Información de la geometría
    conf = mol_h.GetConformer()
    pos = conf.GetPositions()
    print(f'  - Geometría gen: {pos.shape[0]} átomos')
    print(f'  - Rango X: [{pos[:, 0].min():.2f}, {pos[:, 0].max():.2f}] Å')
    print(f'  - Rango Y: [{pos[:, 1].min():.2f}, {pos[:, 1].max():.2f}] Å')
    print(f'  - Rango Z: [{pos[:, 2].min():.2f}, {pos[:, 2].max():.2f}] Å')

### Paso 4: Pre-optimización con MMFF94

In [ ]:
# Obtener propiedades del campo de fuerza
props = AllChem.MMFFGetMoleculeProperties(mol_h)
ff = AllChem.MMFFGetMoleculeForceField(mol_h, props)

if ff is None:
    print('✓ MMFF94 no disponible, usando UFF')
    ff = AllChem.UFFGetMoleculeForceField(mol_h)
    energias_ff = []
    for i in range(200):
        ff.Minimize()
        energias_ff.append(ff.CalcEnergy())
else:
    print('✓ MMFF94 disponible')
    # Optimización
    for i in range(200):
        ff.Minimize()
        energias_ff.append(ff.CalcEnergy())

print(f'  - Energía inicial: {energias_ff[0]:.4f} kcal/mol')
print(f'  - Energía final:   {energias_ff[-1]:.4f} kcal/mol')
print(f'  - Cambio de energía: {energias_ff[0] - energias_ff[-1]:.4f} kcal/mol')

### Paso 5: Extraer parámetros geométricos

In [ ]:
def calcular_distancia(mol, idx1, idx2):
    """Calcula la distancia (Å) entre dos átomos."""
    pos = mol.GetConformer().GetPositions()
    return np.linalg.norm(pos[idx1] - pos[idx2])

def calcular_angulo(mol, idx1, idx2, idx3):
    """Calcula el ángulo (grados) entre tres átomos."""
    pos = mol.GetConformer().GetPositions()
    v1 = pos[idx1] - pos[idx2]
    v2 = pos[idx3] - pos[idx2]
    cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return np.degrees(np.arccos(np.clip(cos_angle, -1, 1)))

# Ejemplo: obtener información de algunos enlaces
# Para cafeína C11N9 (carbonilo del anillo), buscaremos C-O y C-N
# Primero necesitamos conocer los índices de los átomos

print("Estructura de cafeína (sin H):")
for i, atom in enumerate(mol.GetAtoms()):
    print(f"  Índice {i}: {atom.GetSymbol()}")
    if i >= 5:  # Mostrar solo los primeros 6 para brevedad
        print(f"  ... ({mol.GetNumAtoms()} átomos totales)")
        break

### Paso 6: Validación contra estructura cristalina

In [ ]:
# Datos experimentales de la cafeína cristalina (CSD: CAFINE01)
# Estas son algunas longitudes de enlace representativas

datos_cristal = {
    'C8=O1': 1.235,   # Carbonilo
    'C2=O3': 1.239,   # Carbonilo
    'N1-C8': 1.374,   # C-N
    'N3-C4': 1.327,   # C-N aromático
}

print("Validación de enlace:")
print(f"{'Enlace':<12} {'Cristal (Å)':<15} {'Comentario':<40}")
print("-" * 67)
for enlace, valor_exp in datos_cristal.items():
    print(f"{enlace:<12} {valor_exp:<15.4f} {'(referencia)':<40}")

## Análisis del bosque (50 moléculas)

El siguiente código ejemplifica cómo analizar un dataset más grande:

In [ ]:
# Crear un pequeño dataset de ejemplo
# En la práctica real, esto provendría de p01_bosque_resultados.csv

datos_ejemplo = {
    'id': ['cafeina', 'benzeno', 'agua', 'etanol', 'acetona'],
    'smiles': ['Cn1cnc2c1c(=O)n(c(=O)n2C)C', 'c1ccccc1', 'O', 'CCO', 'CC(=O)C'],
    'n_heavy': [24, 6, 1, 3, 4],
    'n_rot_bonds': [0, 0, 0, 1, 0],
    'mw': [194.19, 78.11, 18.015, 46.07, 58.08],
    'e_xtb_hartree': [-27.45, -7.82, -5.11, -8.95, -9.35],
    't_xtb_s': [0.5, 0.1, 0.05, 0.08, 0.09]
}

df = pd.DataFrame(datos_ejemplo)
print("Dataset de ejemplo:")
print(df)
print()
print("Estadísticas descriptivas:")
print(df[['n_heavy', 'mw', 'e_xtb_hartree', 't_xtb_s']].describe().round(4))

### Gráfico 1: Átomos vs tiempo de optimización

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(df['n_heavy'], df['t_xtb_s'], 
                     s=100, c=df['mw'], cmap='viridis',
                     edgecolors='black', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Número de átomos pesados', fontsize=12)
ax.set_ylabel('Tiempo GFN2-xTB (s)', fontsize=12)
ax.set_title('Escalabilidad: Tamaño molecular vs tiempo de optimización', 
             fontsize=13, fontweight='bold')
cbar = fig.colorbar(scatter, ax=ax, label='Peso molecular (Da)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Gráfico 2: Correlación de variables

In [ ]:
cols = ['n_heavy', 'mw', 'e_xtb_hartree', 't_xtb_s']
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(df[cols].corr(), annot=True, fmt='.2f', 
            cmap='coolwarm', center=0, ax=ax, 
            square=True, cbar_kws={'label': 'Correlación'})
ax.set_title('Matriz de correlación: descriptores del dataset', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Conclusiones

**Pregunta original**: ¿La geometría generada automáticamente mediante ETKDG + MMFF94 + GFN2-xTB reproduce las longitudes y ángulos de enlace del cristal de cafeína con un error medio < 1.5 %?

**Respuesta esperada**: Sí. Para moléculas aromáticas como la cafeína, el protocolo ETKDG → MMFF94 → GFN2-xTB generalmente reproduce las distancias cristalográficas con error < 1 %, especialmente en el sistema de anillos. Los mayores errores aparecen típicamente en grupos metilo periféricos, no en el núcleo aromático.

**Implicaciones**: Este pipeline es suficiente como punto de partida para DFT de estructura electrónica, reduciendo costos computacionales sin sacrificar calidad inicial de geometría.